## 0. Notebook Purpose

This notebook handles feature selection, train-test splitting, and data preprocessing.

Decision: All transformations are applied in code and saved to `data/processed/`. Raw data downloaded via `kagglehub` remains untouched. This notebook loads the EDA output and prepares model-ready datasets. Shared utility functions are imported from `src/utils.py`.

In [1]:
# ============================================
# Setup, Imports & Path Configuration
# ============================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import pickle
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ============================================
# Robust Project Root Detection
# ============================================
current_dir = Path.cwd()
if (current_dir / "src").exists() and (current_dir / "notebooks").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "notebooks").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError("Could not detect project root. Ensure 'src/' and 'notebooks/' exist.")

# Define absolute paths
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create directories upfront
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Add project root to Python path for src/ imports
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

# ============================================
# Import Shared Utilities & Reproducibility
# ============================================
from src.utils import evaluate_top_k, ensure_dir

np.random.seed(42)
random.seed(42)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Setup complete. Ready for cleaning/preprocessing.")

Project root: /home/azan_khan/NFL-Prediction-Model
Setup complete. Ready for cleaning/preprocessing.


## 1. Load EDA-Processed Data

Decision: Load the dataset with target variable from `data/processed/` (generated by 01_eda.ipynb). This ensures the cleaning notebook operates on a consistent starting point.

In [2]:
# ============================================
# Load Dataset with Target Variable
# ============================================

df = pd.read_csv(PROJECT_ROOT / "data/processed/team_stats_with_target.csv")

print("Dataset loaded from processed directory.")
print("Shape:", df.shape)

# Assertion: Verify target variable exists and has expected properties
assert "super_bowl_winner" in df.columns, "Target variable missing"
assert df["super_bowl_winner"].sum() == 21, "Unexpected number of Super Bowl winners"

Dataset loaded from processed directory.
Shape: (672, 36)


## 2. Feature Selection and Justification

In this section, we define which variables will be used for modeling.

We construct two feature sets:
- A full feature set that includes all available predictors
- A reduced feature set that removes redundant or outcome-based variables

Decision: Two feature sets enable comparison of model performance while testing whether a more interpretable, leakage-resistant feature set can still identify strong contenders.

In [3]:
# ============================================
# Full Feature Set (All Predictors Except Identifiers)
# ============================================

X_full = df.drop(columns=["super_bowl_winner", "team", "year"])
y = df["super_bowl_winner"]

print("Full feature set shape:", X_full.shape)

# ============================================
# Clean Feature Set (Reduced - Excludes Outcome-Based Variables)
# ============================================

features_clean = df.drop(columns=[
    "super_bowl_winner",
    "team",
    "year",
    "wins",           # Decision: Excluded - too close to outcome
    "win_loss_perc",  # Decision: Excluded - derived from wins
    "points_diff",    # Decision: Excluded - derived from points scored/allowed
    "mov"             # Decision: Excluded - redundant with points_diff
])

X_clean = features_clean

print("Clean feature set shape:", X_clean.shape)

# Assertion: Verify clean set has fewer features than full set
assert X_clean.shape[1] < X_full.shape[1], "Clean set should have fewer features"

Full feature set shape: (672, 33)
Clean feature set shape: (672, 29)


### Feature Selection Rationale

**Full Feature Set**
- Includes nearly all available predictors except identifiers such as `team` and `year`
- Used to observe how a more permissive model behaves

**Clean Feature Set**
- Excludes variables that are either too close to the final outcome or mathematically redundant

Decision: Variables removed from clean feature set:
- `wins` and `win_loss_perc`: Highly outcome-based; including them would make prediction trivial and less meaningful.
- `points_diff` and `mov`: Summary variables derived from scoring outcomes; they overlap heavily with more fundamental features (points scored/allowed).

This clean feature set produces a more honest and interpretable model by relying on core team-performance metrics rather than compressed season summaries.

## 3. Train-Test Split

The dataset is split based on time rather than randomly.

Decision: Time-based split ensures training uses past data and testing uses future data. This avoids data leakage and reflects real-world prediction where future outcomes are unknown.

In [4]:
# ============================================
# Train-Test Split (Time-Based)
# ============================================

# Decision: Split at 2019 to simulate predicting recent seasons from historical data
train = df[df["year"] < 2019]
test = df[df["year"] >= 2019]

# Full feature set split
X_train_full = train.drop(columns=["super_bowl_winner", "team", "year"])
y_train = train["super_bowl_winner"]
X_test_full = test.drop(columns=["super_bowl_winner", "team", "year"])
y_test = test["super_bowl_winner"]

print("Train size (full):", X_train_full.shape)
print("Test size (full):", X_test_full.shape)

# Clean feature set split
X_train_clean = train.drop(columns=[
    "super_bowl_winner", "team", "year",
    "wins", "win_loss_perc", "points_diff", "mov"
])
X_test_clean = test.drop(columns=[
    "super_bowl_winner", "team", "year",
    "wins", "win_loss_perc", "points_diff", "mov"
])

# Assertion: Verify temporal split preserves class distribution reasonably
train_winner_rate = y_train.mean()
test_winner_rate = y_test.mean()
print(f"Train winner rate: {train_winner_rate:.3f}, Test winner rate: {test_winner_rate:.3f}")

# Threshold: 21 winners / 672 total ≈ 0.03125 (3.1%)
assert 0.02 < train_winner_rate < 0.05, f"Train winner rate outside expected range (~3.1%), got {train_winner_rate:.3f}"
assert 0.02 < test_winner_rate < 0.05, f"Test winner rate outside expected range (~3.1%), got {test_winner_rate:.3f}"

Train size (full): (512, 33)
Test size (full): (160, 33)
Train winner rate: 0.031, Test winner rate: 0.031


### Train-Test Split Rationale

- Training set: seasons 2003–2018 (16 seasons × ~32 teams = ~512 samples)
- Test set: seasons 2019–2023 (5 seasons × ~32 teams = ~160 samples)

Decision: This split simulates real-world prediction where models trained on historical data predict future outcomes. Random splitting would leak future information into training and produce optimistically biased results.

## 4. Data Preprocessing

### Preprocessing Overview

Before any model can be fit, the feature matrices must be checked for issues that would prevent successful training.

Decision: Missing values are imputed using training-set medians only. The same training-set statistics are applied to the test set to avoid data leakage. Feature scaling is applied only for Logistic Regression (Random Forest does not require scaling).

In [5]:
# ============================================
# Modeling Imports
# ============================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns

In [6]:
# ============================================
# Check Missing Values in Training Features (Full Set)
# ============================================

missing_train_full = X_train_full.isnull().sum()
missing_train_full = missing_train_full[missing_train_full > 0].sort_values(ascending=False)

print("Columns with missing values in X_train_full:")
print(missing_train_full)

# ============================================
# Check Missing Values in Test Features (Full Set)
# ============================================

missing_test_full = X_test_full.isnull().sum()
missing_test_full = missing_test_full[missing_test_full > 0].sort_values(ascending=False)

print("Columns with missing values in X_test_full:")
print(missing_test_full)

Columns with missing values in X_train_full:
ties    320
mov     192
dtype: int64
Columns with missing values in X_test_full:
mov     128
ties     32
dtype: int64


In [7]:
# ============================================
# Impute Missing Values (Full Feature Set)
# ============================================

# Decision: Use training-set medians only to avoid data leakage
train_medians_full = X_train_full.median(numeric_only=True)

X_train_full = X_train_full.fillna(train_medians_full)
X_test_full = X_test_full.fillna(train_medians_full)

print("Remaining missing values in X_train_full:", X_train_full.isnull().sum().sum())
print("Remaining missing values in X_test_full:", X_test_full.isnull().sum().sum())

# Assertion: Verify no missing values remain after imputation
assert X_train_full.isnull().sum().sum() == 0, "Missing values remain in training set"
assert X_test_full.isnull().sum().sum() == 0, "Missing values remain in test set"

Remaining missing values in X_train_full: 0
Remaining missing values in X_test_full: 0


In [8]:
# ============================================
# Impute Missing Values (Clean Feature Set)
# ============================================

# Decision: Same imputation strategy for clean feature set
train_medians_clean = X_train_clean.median(numeric_only=True)

X_train_clean = X_train_clean.fillna(train_medians_clean)
X_test_clean = X_test_clean.fillna(train_medians_clean)

print("Remaining missing values in X_train_clean:", X_train_clean.isnull().sum().sum())
print("Remaining missing values in X_test_clean:", X_test_clean.isnull().sum().sum())

# Assertion: Verify no missing values remain after imputation
assert X_train_clean.isnull().sum().sum() == 0, "Missing values remain in training set (clean)"
assert X_test_clean.isnull().sum().sum() == 0, "Missing values remain in test set (clean)"

Remaining missing values in X_train_clean: 0
Remaining missing values in X_test_clean: 0


### Missing-Value Handling Before Modeling

Some predictor columns contained missing values, which prevented logistic regression from fitting.

Decision: Missing numeric values were imputed using the median of each feature computed from the training set only. The same training-set medians were then applied to the test set. This approach avoids data leakage while allowing the model to train on a complete feature matrix.

Alternative considered: Mean imputation. Rejected because median is more robust to outliers in sports statistics.

In [9]:
# ============================================
# Scale Features (Full Feature Set) - For Logistic Regression Only
# ============================================

# Decision: Scale only for Logistic Regression; Random Forest does not require scaling
scaler_full = StandardScaler()

X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

print("Full feature set scaled.")
print("Scaled train shape:", X_train_full_scaled.shape)
print("Scaled test shape:", X_test_full_scaled.shape)

# ============================================
# Scale Features (Clean Feature Set) - For Logistic Regression Only
# ============================================

scaler_clean = StandardScaler()

X_train_clean_scaled = scaler_clean.fit_transform(X_train_clean)
X_test_clean_scaled = scaler_clean.transform(X_test_clean)

print("Clean feature set scaled.")
print("Scaled train shape:", X_train_clean_scaled.shape)
print("Scaled test shape:", X_test_clean_scaled.shape)

Full feature set scaled.
Scaled train shape: (512, 33)
Scaled test shape: (160, 33)
Clean feature set scaled.
Scaled train shape: (512, 29)
Scaled test shape: (160, 29)


### Feature Scaling Before Logistic Regression

Logistic regression is sensitive to differences in feature scale. Since the predictor variables in this dataset are measured on very different numerical ranges, the features were standardized before model fitting.

Decision: The scaler was fit on the training data only and then applied to the test data. This avoids data leakage while improving model stability and optimizer convergence.

Alternative considered: MinMax scaling. Rejected because StandardScaler handles outliers better and is the default for logistic regression in scikit-learn.

In [10]:
# ============================================
# Save Preprocessed Data for Modeling Notebook
# ============================================

# Ensure directories exist
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

# Save all preprocessed objects to data/processed/ for use in 03_modeling.ipynb
with open(PROJECT_ROOT / "data/processed/X_train_full.pkl", "wb") as f:
    pickle.dump(X_train_full, f)
with open(PROJECT_ROOT / "data/processed/X_test_full.pkl", "wb") as f:
    pickle.dump(X_test_full, f)
with open(PROJECT_ROOT / "data/processed/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)
with open(PROJECT_ROOT / "data/processed/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

with open(PROJECT_ROOT / "data/processed/X_train_clean.pkl", "wb") as f:
    pickle.dump(X_train_clean, f)
with open(PROJECT_ROOT / "data/processed/X_test_clean.pkl", "wb") as f:
    pickle.dump(X_test_clean, f)

with open(PROJECT_ROOT / "data/processed/X_train_full_scaled.pkl", "wb") as f:
    pickle.dump(X_train_full_scaled, f)
with open(PROJECT_ROOT / "data/processed/X_test_full_scaled.pkl", "wb") as f:
    pickle.dump(X_test_full_scaled, f)
with open(PROJECT_ROOT / "data/processed/X_train_clean_scaled.pkl", "wb") as f:
    pickle.dump(X_train_clean_scaled, f)
with open(PROJECT_ROOT / "data/processed/X_test_clean_scaled.pkl", "wb") as f:
    pickle.dump(X_test_clean_scaled, f)

with open(PROJECT_ROOT / "data/processed/scaler_full.pkl", "wb") as f:
    pickle.dump(scaler_full, f)
with open(PROJECT_ROOT / "data/processed/scaler_clean.pkl", "wb") as f:
    pickle.dump(scaler_clean, f)

# Save metadata for reproducibility
metadata = {
    "train_years": list(range(2003, 2019)),
    "test_years": list(range(2019, 2024)),
    "full_features": list(X_full.columns),
    "clean_features": list(X_clean.columns),
    "imputation_strategy": "median_training_set",
    "scaling_applied": ["LogisticRegression"],
    "random_seed": 42
}

with open(PROJECT_ROOT / "data/processed/preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Preprocessing complete. All objects saved to data/processed/")

# Final assertions
assert Path(PROJECT_ROOT / "data/processed/X_train_clean.pkl").exists(), "Clean training features not saved"
assert Path(PROJECT_ROOT / "data/processed/preprocessing_metadata.json").exists(), "Metadata file not saved"

Preprocessing complete. All objects saved to data/processed/
